In [ ]:
import pandas as pd

# Предположим, ваши два датафрейма сохранены в переменные df1 и df2
# (Замените df1 и df2 на ваши реальные переменные, если они называются иначе)
import pandas as pd
df1 =pd.read_parquet('/content/train-00000-of-00001-35919150bc94e0c2 (1).parquet')
df2 = pd.read_parquet('/content/train-00000-of-00001.parquet')
print(df1)
print(df2)
# 1. Обрабатываем первый файл (1332 строки)
# Выбираем нужные колонки и переименовываем: ru -> instruction, bua -> output
df1_prepared = df1[['ru', 'bua']].rename(
    columns={'ru': 'instruction', 'bua': 'output'}
)

# 2. Обрабатываем второй файл (40979 строк)
# Выбираем нужные колонки и переименовываем: ru -> instruction, bxr -> output
df2_prepared = df2[['ru', 'bxr']].rename(
    columns={'ru': 'instruction', 'bxr': 'output'}
)

# 3. Вертикальное объединение (склеиваем их один под другим)
df_instruct = pd.concat([df1_prepared, df2_prepared], axis=0, ignore_index=True)

# 4. Финальная чистка датасета (удаление пустых строк и дубликатов)
df_instruct = df_instruct.dropna(subset=['instruction', 'output'])
df_instruct['instruction'] = df_instruct['instruction'].astype(str).str.strip()
df_instruct['output'] = df_instruct['output'].astype(str).str.strip()

# Удаляем повторяющиеся пары, чтобы модель не зазубривала одно и то же
df_instruct = df_instruct.drop_duplicates(subset=['instruction', 'output'])
df_instruct = df_instruct.reset_index(drop=True)

print("🎉 Объединение завершено успешно!")
print(f"Итоговый размер общего датасета: {df_instruct.shape[0]} пар.")
print("\nПример собранных данных из начала (Официальные документы):")
print(df_instruct.head(2))
print("\nПример собранных данных из конца (Сказки/Библия):")
print(df_instruct.tail(2))


                                                     ru  \
0                                        ПОСТАНОВЛЕНИЕ.   
1                                          г. Улан-Удэ.   
2     О внесении изменений в постановление Правитель...   
3     В целях приведения правового акта Правительств...   
4     Внести следующие изменения в постановление Пра...   
...                                                 ...   
1327  Внести следующие изменения в Перечень мероприя...   
1328  Строки 1.102 и 1.103 изложить в следующей реда...   
1329  Дополнить строками 1.106 - 1.115 следующего со...   
1330  Настоящее постановление вступает в силу со дня...   
1331  Проект представлен Министерством здравоохранен...   

                                                    bua  
0                                              ТОГТООЛ.  
1                                       Улаан-Үдэ хото.  
2     «Хэлсээнэй сэн хубилгаха (дээшэлүүлхэ) арга бо...  
3     Буряад Уласай Засагай газарай хуулита шиидхэбэ...  
4

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate  peft


In [ ]:
!pip install -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 1. Пути и параметры
model_id = "unsloth/Llama-3.2-1B"
output_dir = "/content/buryat_llama_lora_fp16"

# 2. Форматирование русско-бурятского датасета под шаблон Llama 3 Chat
def format_translation_prompts(batch):
    formatted_texts = []
    for ru_text, bur_text in zip(batch['instruction'], batch['output']):
        text = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            f"{ru_text}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{bur_text}<|eot_id|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

print("Подготовка и форматирование датасета...")
df_clean = df_instruct.dropna(subset=['instruction', 'output']).astype(str)
instruct_dataset = Dataset.from_pandas(df_clean).map(format_translation_prompts, batched=True)

print("Загрузка оригинальной модели и токенизатора в чистом FP16...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# !!! КЛЮЧЕВОЙ МОМЕНТ: Загружаем без quantization_config напрямую в VRAM !!!
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,       # Модель весит 2.3 ГБ, памяти T4 хватит с огромным запасом
    device_map="auto"                # Сажаем строго на GPU
)

# 3. Настройка LoRA-параметров (Rank=64 по вашему ТЗ)
print("Конфигурация LoRA-адаптеров...")
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. Оптимальные параметры обучения в FP16
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,                             # Чистый стабильный fp16
    optim="adamw_torch_fused",             # Быстрый объединенный оптимизатор PyTorch вместо bnb
    gradient_checkpointing=True,
    report_to="none"
)

# 5. Конфигурация SFT (Защита от нулевого лосса)
"""sft_config = SFTConfig(
    max_seq_length=512,
    packing=True,                         # Склеивает строки, убирая скрытые маски
    dataset_text_field="text"
)
"""
# 6. Инициализация тренера
trainer = SFTTrainer(
    model=model,
    train_dataset=instruct_dataset,
    args=training_args,

)

print("\n🚀 Старт инструктивного LoRA обучения в FP16...")
trainer.train()

# 7. Сохранение результатов
print("\nОбучение успешно завершено! Сохранение адаптеров...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Финал! Ваши обученные LoRA-веса сохранены в папку: {output_dir}")



Подготовка и форматирование датасета...


Map:   0%|          | 0/41744 [00:00<?, ? examples/s]

Загрузка оригинальной модели и токенизатора в чистом FP16...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Конфигурация LoRA-адаптеров...


...

Building labels for train dataset:   0%|          | 0/41744 [00:00<?, ? examples/s]


🚀 Старт инструктивного LoRA обучения в FP16...


Step,Training Loss
10,5.788294
20,6.072018
30,5.457313
40,5.331383
50,4.935739
60,4.226405
70,3.996109
80,3.816288
90,3.541368
100,3.882202


Step,Training Loss
10,5.788294
20,6.072018
30,5.457313
40,5.331383
50,4.935739
60,4.226405
70,3.996109
80,3.816288
90,3.541368
100,3.882202


KeyboardInterrupt: 

In [ ]:
# 7. Сохранение результатов
print("\nОбучение успешно завершено! Сохранение адаптеров...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Финал! Ваши обученные LoRA-веса сохранены в папку: {output_dir}")


Обучение успешно завершено! Сохранение адаптеров...
Финал! Ваши обученные LoRA-веса сохранены в папку: /content/buryat_llama_lora_fp16


In [ ]:
from huggingface_hub import HfApi
import os

# 1. Настройки путей
# Укажите имя папки в Colab, где лежат ваши финальные файлы LoRA (например, после Шага 5)
local_folder_path = "/content/buryat_llama_lora_fp16"

# Придумайте название репозитория на Hugging Face
# Формат строго: "ваш_никнейм/название_репозитория"
hf_user_name = "poco28"  # Замените на ваш никнейм на HF
repo_name = "llama-3.2-1b-ru-bua-translator"
repo_id = f"{hf_user_name}/{repo_name}"

# 2. Инициализируем API Hugging Face
api = HfApi()

# 3. Автоматически создаем репозиторий на сайте (если он еще не создан)
print(f"Создание репозитория {repo_id} на Hugging Face...")
try:
    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=True  # private=True делает модель видимой только вам. Поменяйте на False, если хотите открыть её всем.
    )
    print("Репозиторий успешно создан!")
except Exception as e:
    print("Репозиторий уже существует или произошла ошибка, продолжаем загрузку файлов...")

# 4. 🔥 ЗАЛИВАЕМ ВСЕ ФАЙЛЫ ИЗ ПАПКИ ЗА ОДИН РАЗ 🔥
print(f"Начинается загрузка всех файлов из папки '{local_folder_path}'...")
print("Шкала tqdm ниже покажет прогресс для каждого файла...")

api.upload_folder(
    folder_path=local_folder_path,  # Какую локальную папку загружать
    repo_id=repo_id,                # В какой репозиторий отправлять
    repo_type="model",
    commit_message="Initial commit: Ru-Bua LoRA translator adapter and tokenizer config"
)

print(f"\n🎉 Поздравляю! Все файлы успешно загружены.")
print(f"Ваша модель доступна по прямой ссылке: https://huggingface.co{repo_id}")


Создание репозитория poco28/llama-3.2-1b-ru-bua-translator на Hugging Face...
Репозиторий успешно создан!
Начинается загрузка всех файлов из папки '/content/buryat_llama_lora_fp16'...
Шкала tqdm ниже покажет прогресс для каждого файла...

🎉 Поздравляю! Все файлы успешно загружены.
Ваша модель доступна по прямой ссылке: https://huggingface.copoco28/llama-3.2-1b-ru-bua-translator


In [ ]:
from transformers import pipeline

model.eval() # Переводим модель в режим инференса

# Создаем генератор
generator = pipeline(task="text-generation", model=model, tokenizer=tokenizer)

# Запрос на русском языке
ru_prompt = "Переведи на бурятский язык фразу: привет, мой старый друг."

prompt = (
    f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    f"{ru_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

print("Модель переводит...")
with torch.no_grad():
    outputs = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

print("\n--- ОТВЕТ МОДЕЛИ ---")
# Заходим в первый элемент списка [0] и берем значение по ключу 'generated_text'
print(outputs[0]['generated_text'])



[transformers] Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Модель переводит...

--- ОТВЕТ МОДЕЛИ ---
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Переведи на бурятский язык фразу: привет, мой старый друг.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Буряад хэлэн дээрэ хэлэжэ үгэ: „Хүндэтнай намда амар намдаа“ гэжэ.


In [ ]:
import pandas as pd

# 1. Читаем оба файла
df_ru = pd.read_parquet('russian.parquet')   # Файл с русскими фразами
df_bur = pd.read_parquet('buryat.parquet')   # Файл с бурятскими фразами

# 2. Убеждаемся, что колонки называются правильно
# (для нашего кода обучения нужны 'instruction' и 'output')
# Предположим, внутри файлов колонки назывались 'text'. Переименуем их:
df_ru = df_ru.rename(columns={df_ru.columns[0]: 'instruction'})
df_bur = df_bur.rename(columns={df_bur.columns[0]: 'output'})

# 3. Склеиваем их горизонтально (бок о бок) по индексам строк
df_instruct = pd.concat([df_ru.reset_index(drop=True), df_bur.reset_index(drop=True)], axis=1)

# 4. Проверяем результат
print(f"Размер собранного датасета: {df_instruct.shape}")
print(df_instruct.head(3)) # Выведет первые 3 пары строк
